# Rubin/LSST DDF — Observing Cadence Characterisation for Supernovae

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-25
- **purpose** : Prepare figures for the SCOC (Survey Cadence Optimisation Committee) talk on DDF observing strategy

## Purpose

This notebook characterises the **observing cadence** seen by real Rubin/LSST supernovae
detected through the Fink alert broker, using the cached light-curve parquet files
produced by notebooks 01, 02, 03, and 05.

Two families of diagnostics are produced:

1. **Visit-count histograms** — number of visits per band per SN (and aggregated over all SNe).
2. **Inter-visit time histograms** — distribution of time gaps Δt between successive
   observations, regardless of band and per band (binned in days).

Where possible the plots are broken down **per DDF field** using the `r:ra` / `r:dec`
sky coordinates saved in the catalog.  A simple nearest-centre assignment maps each
SN to one of the known Rubin DDF pointings.

### Data sources (light-curve caches)
```
data_NB07_01_TNS_SN/        lc_<diaObjectId>.parquet   (TNS confirmed SNe)
data_NB07_02_SN_NEAR/       lc_<diaObjectId>.parquet   (sn_near_galaxy — NB02)
data_NB07_03_FINK_TNS_SN_FIT_SALT2/  lc_<diaObjectId>.parquet
data_NB07_05_FINK_SN_NEARGAL_FIT_SALT2/light_curves/  lc_<diaObjectId>.parquet
```

### Column conventions
- `r:midpointMjdTai` — MJD of mid-exposure
- `r:band`           — single-letter filter {u, g, r, i, z, y}
- `r:psfFlux`        — PSF flux in nJy
- `r:ra`, `r:dec`    — sky coordinates (available in catalog, not always in LC)

### References
- Rubin SCOC: https://www.lsst.org/scientists/survey-design/scoc
- Fink LSST portal: https://api.lsst.fink-portal.org


## 0 — Imports

In [ ]:
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from astropy.time import Time

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
    }
)
print("Imports OK")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "xx-large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "xx-large",
    "axes.titlesize": "xx-large",
    "xtick.labelsize": "xx-large",
    "ytick.labelsize": "xx-large",
}
plt.rcParams.update(params)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 — Configuration

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB07_06_CADENCE_SN"
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Light-curve cache directories to scan ─────────────────────────────────────
# Priority order: first found cache file wins for a given diaObjectId.
LC_CACHE_DIRS = [
    Path("data_NB07_03_FINK_TNS_SN_FIT_SALT2"),  # NB03 cache (TNS + re-fetched)
    Path("data_NB07_01_TNS_SN"),  # NB01 original cache
    Path("data_NB07_05_FINK_SN_NEARGAL_FIT_SALT2") / "light_curves",  # NB05 cache
    Path("data_NB07_02_SN_NEAR"),  # NB02 cache if present
]

# ── Catalog files (for RA/Dec → DDF assignment) ───────────────────────────────
CATALOG_TNS = Path("data_NB07_01_TNS_SN") / "catalog_fink_in_tns_lsst.parquet"
CATALOG_NEAR = Path("data_NB07_05_FINK_SN_NEARGAL_FIT_SALT2") / "catalog_sn_near_galaxy_full.parquet"

# ── Band settings ─────────────────────────────────────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#1f77b4",  # blue
    "g": "#2ca02c",  # green
    "r": "#d62728",  # red
    "i": "#ff7f0e",  # orange
    "z": "#8c564b",  # brown
    "y": "#9467bd",  # violet
}

# ── Rubin Deep Drilling Fields (RA, Dec, name) ────────────────────────────────
# Source: Ivezic et al. 2019 / SCOC documentation
DDF_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "XMM-LSS": (34.4757, -5.0900),
    "ELAIS-S1": (9.4500, -44.0000),
    "ECDFS": (53.1250, -28.1000),
    "EDFS-a": (58.9000, -49.3150),
    "EDFS-b": (63.6000, -47.6000),
}
DDF_RADIUS_DEG = 3.0  # matching radius for DDF assignment

# ── Histogram binning ─────────────────────────────────────────────────────────
DT_BIN_DAYS = 1.0  # inter-visit time bin width in days
DT_MAX_DAYS = 30.0  # maximum Δt shown in the histogram

print("Configuration OK")
print(f"Output figures → {FIGS_DIR}/")
print(f"DDF fields     : {list(DDF_FIELDS.keys())}")

## 2 — Load light curves from cache

We scan all configured cache directories for `lc_<diaObjectId>.parquet` files
and load each unique object once.

In [ ]:
def find_lc_files(cache_dirs: list) -> dict:
    """Scan cache directories and return a mapping diaObjectId -> Path.

    The first directory in the list that contains lc_<oid>.parquet wins
    (priority order matters).
    """
    oid_to_path: dict[int, Path] = {}
    for cdir in cache_dirs:
        if not cdir.is_dir():
            continue
        for fpath in sorted(cdir.glob("lc_*.parquet")):
            stem = fpath.stem  # e.g. "lc_170019717277810735"
            if not stem.startswith("lc_"):
                continue
            try:
                oid = int(stem[3:])
            except ValueError:
                continue
            if oid not in oid_to_path:  # first hit wins
                oid_to_path[oid] = fpath
    return oid_to_path


oid_to_path = find_lc_files(LC_CACHE_DIRS)
print(f"Found {len(oid_to_path)} unique diaObjectId light-curve files.")
for oid, p in list(oid_to_path.items())[:5]:
    print(f"  {oid}  ←  {p}")

In [ ]:
def load_lc(path: Path) -> pd.DataFrame:
    """Load a light-curve parquet file and normalise column names.

    Returns a DataFrame with at least:
        mjd   : float MJD (from r:midpointMjdTai)
        band  : str single-letter filter
        flux  : float psfFlux [nJy]
        fluxerr : float psfFluxErr [nJy]
        snr   : float
    """
    df = pd.read_parquet(path)
    rename = {
        "r:midpointMjdTai": "mjd",
        "r:band": "band",
        "r:psfFlux": "flux",
        "r:psfFluxErr": "fluxerr",
        "r:snr": "snr",
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    # Normalise band to single lowercase letter
    if "band" in df.columns:
        df["band"] = df["band"].str.strip().str.lower()
    return df.sort_values("mjd").reset_index(drop=True) if "mjd" in df.columns else df


# Load all light curves
lc_dict: dict[int, pd.DataFrame] = {}
for oid, path in oid_to_path.items():
    try:
        lc = load_lc(path)
        if not lc.empty and "mjd" in lc.columns and "band" in lc.columns:
            lc_dict[oid] = lc
    except Exception as exc:
        print(f"  [warn] {oid}: {exc}")

print(f"Successfully loaded {len(lc_dict)} light curves.")

## 3 — Assign DDF field to each SN

We try to recover RA/Dec from the catalog files.
If unavailable, we try to extract it from the LC dataframes directly (when `r:ra` is present).
Objects further than `DDF_RADIUS_DEG` from any DDF centre are labelled `"WFD/other"`.

In [ ]:
def load_radec_catalog(path: Path, oid_col: str = "r:diaObjectId") -> pd.DataFrame:
    """Load a catalog parquet and return a (diaObjectId, ra, dec) DataFrame."""
    if not path.exists():
        return pd.DataFrame(columns=["diaObjectId", "ra", "dec"])
    cat = pd.read_parquet(path)
    rename = {oid_col: "diaObjectId", "r:ra": "ra", "r:dec": "dec"}
    cat = cat.rename(columns={k: v for k, v in rename.items() if k in cat.columns})
    needed = [c for c in ["diaObjectId", "ra", "dec"] if c in cat.columns]
    if "diaObjectId" not in needed:
        return pd.DataFrame(columns=["diaObjectId", "ra", "dec"])
    cat = cat[needed].drop_duplicates(subset="diaObjectId")
    return cat


# Build RA/Dec table from catalog files
radec_tns = load_radec_catalog(CATALOG_TNS)
radec_near = load_radec_catalog(CATALOG_NEAR)
radec_all = pd.concat([radec_tns, radec_near], ignore_index=True)
radec_all = radec_all.drop_duplicates(subset="diaObjectId").reset_index(drop=True)

# Also try to extract RA/Dec from LC files directly (some have r:ra / r:dec)
extra_rows = []
for oid, lc in lc_dict.items():
    if oid in radec_all["diaObjectId"].values:
        continue
    for col_ra, col_dec in [("r:ra", "r:dec"), ("ra", "dec")]:
        if col_ra in lc.columns and col_dec in lc.columns:
            ra = lc[col_ra].dropna().iloc[0] if not lc[col_ra].dropna().empty else np.nan
            dec = lc[col_dec].dropna().iloc[0] if not lc[col_dec].dropna().empty else np.nan
            extra_rows.append({"diaObjectId": oid, "ra": ra, "dec": dec})
            break

if extra_rows:
    radec_all = pd.concat([radec_all, pd.DataFrame(extra_rows)], ignore_index=True)

radec_map = radec_all.set_index("diaObjectId")[["ra", "dec"]].to_dict("index")
print(f"RA/Dec available for {len(radec_map)} / {len(lc_dict)} objects.")

- Note it may be possible dia-objects are duplicated in the different sources of SN.

In [ ]:
def assign_ddf(
    ra: float, dec: float, ddf_fields: dict = DDF_FIELDS, radius_deg: float = DDF_RADIUS_DEG
) -> str:
    """Return the name of the nearest DDF field within radius_deg, or 'WFD/other'."""
    if not (np.isfinite(ra) and np.isfinite(dec)):
        return "Unknown"
    best_name = "WFD/other"
    best_dist = radius_deg
    for name, (ra0, dec0) in ddf_fields.items():
        # Approximate great-circle distance
        dra = (ra - ra0) * np.cos(np.radians((dec + dec0) / 2.0))
        ddec = dec - dec0
        dist = np.sqrt(dra**2 + ddec**2)
        if dist < best_dist:
            best_dist = dist
            best_name = name
    return best_name


# Build diaObjectId → DDF name mapping
oid_to_ddf: dict[int, str] = {}
for oid in lc_dict:
    coords = radec_map.get(oid, {})
    ra = coords.get("ra", np.nan)
    dec = coords.get("dec", np.nan)
    oid_to_ddf[oid] = assign_ddf(
        float(ra) if pd.notna(ra) else np.nan, float(dec) if pd.notna(dec) else np.nan
    )

from collections import Counter

ddf_counts = Counter(oid_to_ddf.values())
print("DDF assignment summary:")
for ddf_name, cnt in sorted(ddf_counts.items()):
    print(f"  {ddf_name:15s} : {cnt} SNe")

## 4 — Build visit-count and inter-visit-time tables

For each SN we compute:
- **nvisits_per_band** : dict band → int
- **dt_all** : sorted array of all inter-visit time gaps Δt (days), regardless of band
- **dt_per_band** : dict band → sorted array of Δt within that band

In [ ]:
def compute_cadence(lc: pd.DataFrame) -> dict:
    """Compute visit counts and inter-visit time gaps for one SN light curve.

    Parameters
    ----------
    lc : DataFrame with at least columns 'mjd' and 'band'

    Returns
    -------
    dict with keys:
        nvisits        : int    total number of visits
        nvisits_band   : dict band -> int
        dt_all         : ndarray  inter-visit gaps in days (any band)
        dt_band        : dict band -> ndarray of intra-band gaps
        t_min, t_max   : float  MJD range
        duration       : float  t_max - t_min  (days)
    """
    mjd_all = np.sort(lc["mjd"].dropna().values)
    dt_all = np.diff(mjd_all) if len(mjd_all) > 1 else np.array([])

    nvisits_band = {}
    dt_band = {}
    for b in BAND_ORDER:
        mjd_b = np.sort(lc.loc[lc["band"] == b, "mjd"].dropna().values)
        nvisits_band[b] = len(mjd_b)
        dt_band[b] = np.diff(mjd_b) if len(mjd_b) > 1 else np.array([])

    t_min = float(mjd_all.min()) if len(mjd_all) else np.nan
    t_max = float(mjd_all.max()) if len(mjd_all) else np.nan

    return {
        "nvisits": len(mjd_all),
        "nvisits_band": nvisits_band,
        "dt_all": dt_all,
        "dt_band": dt_band,
        "t_min": t_min,
        "t_max": t_max,
        "duration": t_max - t_min if (np.isfinite(t_min) and np.isfinite(t_max)) else np.nan,
    }


# Compute cadence statistics for every SN
cadence: dict[int, dict] = {}
for oid, lc in lc_dict.items():
    cadence[oid] = compute_cadence(lc)

print(f"Cadence computed for {len(cadence)} SNe.")

# Build a summary DataFrame
rows = []
for oid, cad in cadence.items():
    row = {
        "diaObjectId": oid,
        "ddf": oid_to_ddf.get(oid, "Unknown"),
        "nvisits": cad["nvisits"],
        "duration": cad["duration"],
        "t_min": cad["t_min"],
        "t_max": cad["t_max"],
    }
    for b in BAND_ORDER:
        row[f"n_{b}"] = cad["nvisits_band"][b]
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_parquet(DATA_DIR / "cadence_summary.parquet", index=False)
print(summary_df.describe())

In [ ]:
# dump one  entry for the cadence dictionnary corresponding on one SN curves
the_index_selected = 10
cadence[[list(cadence.keys())][0][the_index_selected]]

## 5 — Histograms: number of visits per band per SN

### 5.1 Global (all DDFs merged)

In [ ]:
# compute myself the ranges
max_visits = {}
min_visits = {}
mean_visits = {}

# loop on bands
for b in BAND_ORDER:
    list_if_counts_inband = []

    # loop per SN event
    for cad in cadence.values():
        n_in_band = len(cad["dt_band"][b])
        list_if_counts_inband.append(n_in_band)
    list_if_counts_inband = np.array(list_if_counts_inband)
    max_visits[b] = np.max(list_if_counts_inband)
    min_visits[b] = np.min(list_if_counts_inband)
    mean_visits[b] = np.mean(list_if_counts_inband)
# max(max(cad["nvisits_band"][b] for cad in cadence.values()) for b in BAND_ORDER)

In [ ]:
print("max_visits:", max_visits)
print("min_visits:", min_visits)
print("mean_visits:", mean_visits)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=False, sharey=False)
fig.suptitle(f"Number of visits per band per SN — all fields ({len(cadence)} SNe)", fontsize=13, y=1.01)

# max_visits = max(max(cad["nvisits_band"][b] for cad in cadence.values()) for b in BAND_ORDER)
# bins_nv = np.arange(0, max_visits + 2) - 0.5  # integer bins

for k, b in enumerate(BAND_ORDER):
    ax = axes[k // 3][k % 3]
    counts = [cad["nvisits_band"][b] for cad in cadence.values()]
    n_nonzero = sum(c > 0 for c in counts)
    bins_nv = np.arange(0, max_visits[b] + 2) - 0.5  # integer bins

    if max_visits[b] < 100:
        ax.hist(
            counts,
            bins=bins_nv,
            # bins=50,
            color=BAND_COLORS[b],
            edgecolor="white",
            linewidth=0.6,
        )
    else:
        ax.hist(
            counts,
            # bins=bins_nv,
            bins=50,
            color=BAND_COLORS[b],
            edgecolor="white",
            linewidth=0.6,
        )

    ax.axvline(
        np.median([c for c in counts if c > 0]) if n_nonzero > 0 else 0,
        color="k",
        ls="--",
        lw=1.5,
        label="median (>0)",
    )
    ax.set_title(f"Band {b}  (N covered = {n_nonzero}/{len(counts)})")
    ax.set_xlabel(f"N visits in band {b}")
    ax.set_ylabel("Number of SNe")
    ax.legend(fontsize=10)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIGS_DIR / "nvisits_per_band_all.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/nvisits_per_band_all.png")
plt.show()

### 5.2 Stacked bar: mean visits per band, split by DDF

In [ ]:
# Group by DDF
ddfs_present = [d for d in summary_df["ddf"].unique() if d not in ("WFD/other", "Unknown")]
ddfs_present_all = list(summary_df["ddf"].unique())

print("DDFs with at least one SN:", ddfs_present_all)

if len(ddfs_present_all) > 1:
    fig, ax = plt.subplots(figsize=(max(7, 2 * len(ddfs_present_all)), 5))
    x = np.arange(len(ddfs_present_all))
    bottom = np.zeros(len(ddfs_present_all))

    for b in BAND_ORDER:
        means = [summary_df.loc[summary_df["ddf"] == ddf, f"n_{b}"].mean() for ddf in ddfs_present_all]
        ax.bar(
            x, means, bottom=bottom, color=BAND_COLORS[b], label=f"band {b}", edgecolor="white", linewidth=0.4
        )
        bottom += np.array(means)

    # Overlay total number of SNe
    for i, ddf in enumerate(ddfs_present_all):
        n_sn = (summary_df["ddf"] == ddf).sum()
        ax.text(i, bottom[i] + 0.3, f"N={n_sn}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(ddfs_present_all, rotation=25, ha="right")
    ax.set_ylabel("Mean number of visits per SN")
    ax.set_title("Mean visit count per band per SN, split by DDF")
    ax.legend(ncol=6, fontsize=9, loc="upper left")
    ax.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "nvisits_per_band_by_ddf_stacked.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved → {FIGS_DIR}/nvisits_per_band_by_ddf_stacked.png")
    plt.show()
else:
    print("Only one DDF present — skipping DDF-split stacked bar (see per-SN histograms above).")

### 5.3 Per-SN summary: sorted bar chart (total visits)

In [ ]:
# Sort SNe by total visit count
summary_sorted = summary_df.sort_values("nvisits", ascending=False).reset_index(drop=True)
n_sn_plot = len(summary_sorted)

fig, ax = plt.subplots(figsize=(max(8, 0.35 * n_sn_plot + 2), 5))
x = np.arange(n_sn_plot)
bottom = np.zeros(n_sn_plot)

for b in BAND_ORDER:
    vals = summary_sorted[f"n_{b}"].values.astype(float)
    ax.bar(x, vals, bottom=bottom, color=BAND_COLORS[b], label=f"{b}", edgecolor="white", linewidth=0.3)
    bottom += vals

# DDF labels at the top
for i, row in summary_sorted.iterrows():
    ax.text(
        i, bottom[i] + 0.5, row["ddf"][:6], ha="center", va="bottom", fontsize=6, rotation=70, color="dimgrey"
    )

ax.set_xlabel("SN index (sorted by total visits)")
ax.set_ylabel("Total number of visits")
ax.set_title(f"Per-SN visit count by band ({n_sn_plot} SNe from Fink/LSST)")
ax.legend(ncol=6, fontsize=9)
ax.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(FIGS_DIR / "nvisits_per_sn_sorted.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/nvisits_per_sn_sorted.png")
plt.show()

## 6 — Inter-visit time histograms

### 6.1 Global Δt (all bands merged), all DDFs

In [ ]:
# Collect all Δt values across all SNe
dt_all_global = np.concatenate([cad["dt_all"] for cad in cadence.values() if len(cad["dt_all"]) > 0])

bins_dt = np.arange(0, DT_MAX_DAYS + DT_BIN_DAYS, DT_BIN_DAYS)
# bins_dt = np.arange(0, 7 + DT_BIN_DAYS, DT_BIN_DAYS/2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    dt_all_global,
    bins=bins_dt,
    color="steelblue",
    edgecolor="white",
    linewidth=0.5,
)
ax.set_xlabel(r"$\Delta t$ between consecutive visits (any band) [days]")
ax.set_ylabel("Count (all SNe combined)")
ax.set_title(
    f"Inter-visit time distribution (all bands, all DDFs)\n"
    f"N_gaps = {len(dt_all_global)}  |  median = {np.median(dt_all_global):.2f} d  "
    f"|  mean = {np.mean(dt_all_global):.2f} d"
)
ax.set_xlim(0, DT_MAX_DAYS)
# ax.set_xlim(0, 7)
ax.set_yscale("log")
ax.axvline(
    np.median(dt_all_global), color="k", ls="--", lw=1.5, label=f"median = {np.median(dt_all_global):.1f} d"
)
ax.axvline(
    np.mean(dt_all_global), color="tomato", ls=":", lw=1.5, label=f"mean = {np.mean(dt_all_global):.1f} d"
)
ax.legend()
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIGS_DIR / "dt_all_bands_global.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/dt_all_bands_global.png")
plt.show()

### 6.2 Δt per band (6-panel grid)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
fig.suptitle("Intra-band inter-visit time distribution — all DDFs", fontsize=13, y=1.01)

for k, b in enumerate(BAND_ORDER):
    ax = axes[k // 3][k % 3]
    dt_b = np.concatenate([cad["dt_band"][b] for cad in cadence.values() if len(cad["dt_band"][b]) > 0])
    if len(dt_b) == 0:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")
        ax.set_title(f"Band {b}")
        continue

    ax.hist(
        dt_b,
        bins=bins_dt,
        color=BAND_COLORS[b],
        edgecolor="white",
        linewidth=0.5,
    )
    med_b = np.median(dt_b)
    mea_b = np.mean(dt_b)
    ax.axvline(med_b, color="k", ls="--", lw=1.5, label=f"median={med_b:.1f}d")
    ax.axvline(mea_b, color="tomato", ls=":", lw=1.5, label=f"mean={mea_b:.1f}d")
    ax.set_title(f"Band {b}  (N_gaps = {len(dt_b)})")
    ax.set_xlabel(r"$\Delta t$ [days]")
    ax.set_ylabel("Count")
    ax.set_yscale("log")
    ax.set_xlim(0, DT_MAX_DAYS)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIGS_DIR / "dt_per_band_global.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/dt_per_band_global.png")
plt.show()

### 6.3 Δt per DDF (all bands merged) — if multiple DDFs detected

In [ ]:
unique_ddfs = sorted(set(oid_to_ddf.values()))
n_ddfs = len(unique_ddfs)

if n_ddfs > 1:
    ncols_ddf = min(3, n_ddfs)
    nrows_ddf = math.ceil(n_ddfs / ncols_ddf)
    fig, axes = plt.subplots(nrows_ddf, ncols_ddf, figsize=(5 * ncols_ddf, 4 * nrows_ddf), sharey=False)
    if n_ddfs == 1:
        axes = np.array([[axes]])
    elif nrows_ddf == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle("Inter-visit time per DDF (all bands merged)", fontsize=13, y=1.01)

    for idx, ddf_name in enumerate(unique_ddfs):
        ax = axes[idx // ncols_ddf][idx % ncols_ddf]
        oids_ddf = [oid for oid, d in oid_to_ddf.items() if d == ddf_name]
        dt_ddf = (
            np.concatenate([cadence[oid]["dt_all"] for oid in oids_ddf if len(cadence[oid]["dt_all"]) > 0])
            if oids_ddf
            else np.array([])
        )

        if len(dt_ddf) == 0:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
            ax.set_title(ddf_name)
            continue

        ax.hist(dt_ddf, bins=bins_dt, color="steelblue", edgecolor="white", linewidth=0.5)
        med = np.median(dt_ddf)
        mea = np.mean(dt_ddf)
        ax.axvline(med, color="k", ls="--", lw=1.5, label=f"median={med:.1f}d")
        ax.axvline(mea, color="tomato", ls=":", lw=1.5, label=f"mean={mea:.1f}d")
        ax.set_title(f"{ddf_name}  (N_SN={len(oids_ddf)}, N_gaps={len(dt_ddf)})")
        ax.set_xlabel(r"$\Delta t$ [days]")
        ax.set_ylabel("Count")
        ax.set_xlim(0, DT_MAX_DAYS)
        ax.set_yscale("log")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)

    # Hide empty panels
    for idx in range(n_ddfs, nrows_ddf * ncols_ddf):
        axes[idx // ncols_ddf][idx % ncols_ddf].set_visible(False)

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "dt_all_bands_per_ddf.png", dpi=150, bbox_inches="tight")
    print(f"Figure saved → {FIGS_DIR}/dt_all_bands_per_ddf.png")
    plt.show()
else:
    print(f"Only one DDF label found ({unique_ddfs[0]}) — skipping per-DDF Δt panels.")

### 6.4 Per-band Δt, split by DDF

One sub-figure per DDF, each with a 2×3 grid of band panels.

In [ ]:
# loop on ddf
for ddf_name in unique_ddfs:
    oids_ddf = [oid for oid, d in oid_to_ddf.items() if d == ddf_name]
    if not oids_ddf:
        continue

    cadence_data = [cadence[oid]["dt_band"][b] for oid in oids_ddf if len(cadence[oid]["dt_band"][b]) > 0]

    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
    fig.suptitle(f"Intra-band inter-visit time — {ddf_name}  (N_SN = {len(oids_ddf)})", fontsize=12, y=1.01)

    for k, b in enumerate(BAND_ORDER):
        ax = axes[k // 3][k % 3]

        # check if that ddf has data
        cadence_data_b = [
            cadence[oid]["dt_band"][b] for oid in oids_ddf if len(cadence[oid]["dt_band"][b]) > 0
        ]
        if len(cadence_data_b) == 0:
            print(f"No data in band {b} inside {ddf_name}")
            continue

        dt_b = (
            np.concatenate(
                [cadence[oid]["dt_band"][b] for oid in oids_ddf if len(cadence[oid]["dt_band"][b]) > 0]
            )
            if oids_ddf
            else np.array([])
        )

        if len(dt_b) == 0:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
            ax.set_title(f"Band {b}")
            continue

        ax.hist(dt_b, bins=bins_dt, color=BAND_COLORS[b], edgecolor="white", linewidth=0.5)
        med_b = np.median(dt_b)
        mea_b = np.mean(dt_b)
        ax.axvline(med_b, color="k", ls="--", lw=1.5, label=f"median={med_b:.1f}d")
        ax.axvline(mea_b, color="tomato", ls=":", lw=1.5, label=f"mean={mea_b:.1f}d")
        ax.set_title(f"Band {b}  (N_gaps = {len(dt_b)})")
        ax.set_xlabel(r"$\Delta t$ [days]")
        ax.set_ylabel("Count")
        ax.set_xlim(0, DT_MAX_DAYS)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)
        ax.set_yscale("log")

    fig.tight_layout()
    fname = FIGS_DIR / f"dt_per_band_{ddf_name.replace('/', '_').replace(' ', '_')}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"Figure saved → {fname}")
    plt.show()

## 7 — Summary statistics table

Printed as a human-readable table: per DDF, per band, mean/median number of visits
and mean/median inter-visit time.

In [ ]:
print(
    f"{'DDF':15s} {'Band':5s} {'N_SN':>6s} {'Mean N_vis':>10s} {'Med N_vis':>10s} "
    f"{'Mean Δt (d)':>11s} {'Med Δt (d)':>11s}"
)
print("-" * 80)

for ddf_name in unique_ddfs:
    oids_ddf = [oid for oid, d in oid_to_ddf.items() if d == ddf_name]
    if not oids_ddf:
        continue
    for b in BAND_ORDER:
        n_vis = [cadence[oid]["nvisits_band"][b] for oid in oids_ddf]

        # check if that ddf has data
        cadence_data_b = [
            cadence[oid]["dt_band"][b] for oid in oids_ddf if len(cadence[oid]["dt_band"][b]) > 0
        ]
        if len(cadence_data_b) == 0:
            print(f"No data in band {b} inside {ddf_name}")

        if len(cadence_data_b) > 0:
            dt_b = (
                np.concatenate(
                    [cadence[oid]["dt_band"][b] for oid in oids_ddf if len(cadence[oid]["dt_band"][b]) > 0]
                )
                if oids_ddf
                else np.array([])
            )
        else:
            dt_b = np.array([])

        mean_nv = np.mean(n_vis)
        med_nv = np.median(n_vis)
        mean_dt = np.mean(dt_b) if len(dt_b) > 0 else np.nan
        med_dt = np.median(dt_b) if len(dt_b) > 0 else np.nan

        print(
            f"{ddf_name:15s} {b:5s} {len(oids_ddf):>6d} "
            f"{mean_nv:>10.1f} {med_nv:>10.1f} "
            f"{mean_dt:>11.2f} {med_dt:>11.2f}"
        )
    print()

## 8 — Individual light-curve cadence overview

For each SN: a panel showing the observation epochs per band as tick marks
(no flux, just timing).  This is a compact SCOC-style "sampling map".

In [ ]:
def plot_sampling_map(ax, oid: int, lc: pd.DataFrame, ddf: str):
    """Show observation epochs per band as a ticker-tape diagram.

    Each band occupies one horizontal lane.  Individual visits are shown
    as vertical tick marks.

    Parameters
    ----------
    ax   : matplotlib Axes
    oid  : diaObjectId label
    lc   : light-curve DataFrame (must contain 'mjd' and 'band')
    ddf  : DDF label for the title
    """
    t_min = lc["mjd"].min()
    t_max = lc["mjd"].max()

    for k, b in enumerate(BAND_ORDER):
        mjd_b = lc.loc[lc["band"] == b, "mjd"].values
        if len(mjd_b) == 0:
            continue
        y_lane = len(BAND_ORDER) - 1 - k  # top = u, bottom = y
        ax.vlines(mjd_b - t_min, ymin=y_lane - 0.35, ymax=y_lane + 0.35, colors=BAND_COLORS[b], linewidth=1.5)

    # Band labels on the y-axis
    ax.set_yticks(range(len(BAND_ORDER)))
    ax.set_yticklabels(BAND_ORDER[::-1], fontsize=8)
    ax.set_xlabel(r"$t - t_{\rm min}$ [days]", fontsize=8)
    ax.set_title(f"diaObj {oid}  |  {ddf}\nN_vis = {len(lc)}  |  dur = {t_max - t_min:.0f} d", fontsize=7)
    ax.set_xlim(-1, t_max - t_min + 1)
    ax.set_ylim(-0.6, len(BAND_ORDER) - 0.4)
    ax.grid(True, axis="x", alpha=0.2)


# Plot all SNe in a grid
n_sn = len(lc_dict)
NCOLS_SAMP = 3
nrows_samp = math.ceil(n_sn / NCOLS_SAMP)

fig, axes = plt.subplots(nrows_samp, NCOLS_SAMP, figsize=(6 * NCOLS_SAMP, 2.8 * nrows_samp), squeeze=False)
fig.suptitle(f"Observation sampling maps — {n_sn} Fink/LSST SNe (Rubin DDF cadence)", fontsize=13, y=1.01)

for k, (oid, lc) in enumerate(sorted(lc_dict.items())):
    ax = axes[k // NCOLS_SAMP][k % NCOLS_SAMP]
    plot_sampling_map(ax, oid, lc, oid_to_ddf.get(oid, "?"))

# Hide empty panels
for k in range(n_sn, nrows_samp * NCOLS_SAMP):
    axes[k // NCOLS_SAMP][k % NCOLS_SAMP].set_visible(False)

fig.tight_layout()
fig.savefig(FIGS_DIR / "sampling_maps_all_sn.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/sampling_maps_all_sn.png")
plt.show()

## 9 — Cadence overview: inter-visit time CDF (cumulative distribution)

A cumulative fraction plot shows what fraction of visits are separated by
less than Δt days.  Useful for SCOC discussions on revisit rate requirements.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# ── Left: CDF per band ───────────────────────────────────────────────────────
ax = axes[0]
for b in BAND_ORDER:
    dt_b = np.concatenate([cad["dt_band"][b] for cad in cadence.values() if len(cad["dt_band"][b]) > 0])
    if len(dt_b) == 0:
        continue
    dt_sorted = np.sort(dt_b)
    cdf = np.arange(1, len(dt_sorted) + 1) / len(dt_sorted)
    ax.plot(dt_sorted, cdf, color=BAND_COLORS[b], lw=2, label=f"{b} (N={len(dt_b)})")

ax.set_xlabel(r"$\Delta t$ [days]")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of inter-visit time, per band (all DDFs)")
ax.set_xlim(0, DT_MAX_DAYS)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="grey", ls=":", lw=1)
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.25)

# ── Right: CDF per DDF (all bands) ────────────────────────────────────────────
ax = axes[1]
cmap_ddf = plt.get_cmap("tab10")
for idx_d, ddf_name in enumerate(unique_ddfs):
    oids_ddf = [oid for oid, d in oid_to_ddf.items() if d == ddf_name]
    dt_ddf = (
        np.concatenate([cadence[oid]["dt_all"] for oid in oids_ddf if len(cadence[oid]["dt_all"]) > 0])
        if oids_ddf
        else np.array([])
    )
    if len(dt_ddf) == 0:
        continue
    dt_sorted = np.sort(dt_ddf)
    cdf = np.arange(1, len(dt_sorted) + 1) / len(dt_sorted)
    ax.plot(
        dt_sorted,
        cdf,
        color=cmap_ddf(idx_d / max(len(unique_ddfs) - 1, 1)),
        lw=2,
        label=f"{ddf_name} (N_SN={len(oids_ddf)})",
    )

ax.set_xlabel(r"$\Delta t$ [days]")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of inter-visit time per DDF (all bands merged)")
ax.set_xlim(0, DT_MAX_DAYS)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="grey", ls=":", lw=1)
ax.legend(ncol=1, fontsize=8)
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIGS_DIR / "dt_cdf_band_and_ddf.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/dt_cdf_band_and_ddf.png")
plt.show()

## 10 — Phase coverage at peak

For cosmological SN analyses, the cadence near peak matters most.
Here we use the epoch of maximum flux in any band as a proxy for $t_0$
and compute Δt values relative to that peak.

In [ ]:
PHASE_WINDOW_PRE = 20.0  # days before peak
PHASE_WINDOW_POST = 40.0  # days after peak

nvisits_near_peak = {}  # oid -> dict band -> int (visits within phase window)

for oid, lc in lc_dict.items():
    # Find rough t0 = epoch of peak psfFlux in any band
    if "flux" not in lc.columns or lc["flux"].dropna().empty:
        continue
    t0_peak = float(lc.loc[lc["flux"].idxmax(), "mjd"])
    mask_peak = (lc["mjd"] >= t0_peak - PHASE_WINDOW_PRE) & (lc["mjd"] <= t0_peak + PHASE_WINDOW_POST)
    lc_peak = lc[mask_peak]
    band_counts = {b: int((lc_peak["band"] == b).sum()) for b in BAND_ORDER}
    nvisits_near_peak[oid] = band_counts

# Build summary per band
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=False)
fig.suptitle(
    f"Visits within [{PHASE_WINDOW_PRE:.0f} d before, {PHASE_WINDOW_POST:.0f} d after] flux peak",
    fontsize=12,
    y=1.01,
)

for k, b in enumerate(BAND_ORDER):
    ax = axes[k // 3][k % 3]
    counts_peak = [nvisits_near_peak[oid][b] for oid in nvisits_near_peak]
    n_nonzero = sum(c > 0 for c in counts_peak)
    bins_nv2 = np.arange(0, max(counts_peak, default=1) + 2) - 0.5
    if len(bins_nv2) < 50:
        ax.hist(counts_peak, bins=bins_nv2, color=BAND_COLORS[b], edgecolor="white", linewidth=0.6)
    else:
        ax.hist(counts_peak, bins=50, color=BAND_COLORS[b], edgecolor="white", linewidth=0.6)
    med_peak = np.median([c for c in counts_peak if c > 0]) if n_nonzero > 0 else 0
    ax.axvline(med_peak, color="k", ls="--", lw=1.5, label=f"median={med_peak:.0f}")
    ax.set_title(f"Band {b}  (covered: {n_nonzero}/{len(counts_peak)})")
    ax.set_xlabel(f"N visits near peak, band {b}")
    ax.set_ylabel("Number of SNe")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIGS_DIR / "nvisits_near_peak_per_band.png", dpi=150, bbox_inches="tight")
print(f"Figure saved → {FIGS_DIR}/nvisits_near_peak_per_band.png")
plt.show()

## 11 — Save cadence summary

In [ ]:
# Add DDF column from oid_to_ddf
summary_df["ddf"] = summary_df["diaObjectId"].map(oid_to_ddf).fillna("Unknown")

# Add peak-window visit counts
for b in BAND_ORDER:
    summary_df[f"n_{b}_peak"] = summary_df["diaObjectId"].apply(
        lambda oid: nvisits_near_peak.get(oid, {}).get(b, 0)
    )

out_parquet = DATA_DIR / "cadence_summary_full.parquet"
out_csv = DATA_DIR / "cadence_summary_full.csv"
summary_df.to_parquet(out_parquet, index=False)
summary_df.to_csv(out_csv, index=False)

print(f"Cadence summary saved:\n  {out_parquet}\n  {out_csv}")
print(f"\nAll figures saved to {FIGS_DIR}/")
summary_df

---
## Notes for the SCOC talk

| Figure | Key message |
|--------|-------------|
| `nvisits_per_band_all.png` | Distribution of visits per SN per band — reveals which filters are well sampled |
| `nvisits_per_band_by_ddf_stacked.png` | Mean cadence per filter per DDF — direct SCOC metric |
| `nvisits_per_sn_sorted.png` | Total visit spread across the SN sample |
| `dt_all_bands_global.png` | Global revisit rate: median and mean Δt |
| `dt_per_band_global.png` | Per-filter revisit cadence |
| `dt_all_bands_per_ddf.png` | DDF-to-DDF comparison of revisit rate |
| `dt_cdf_band_and_ddf.png` | CDF plots — fraction of visits within Δt days |
| `sampling_maps_all_sn.png` | Temporal sampling structure per SN — "ticker tape" view |
| `nvisits_near_peak_per_band.png` | Coverage at peak — critical for SALT2 quality |

**Caveat**: the DDF assignment here relies on RA/Dec availability in the cached
catalogs.  If `r:ra` / `r:dec` were not stored in the LC files or catalog parquets,
all objects will fall under `Unknown` or `WFD/other`.  In that case the global
panels (sections 5–6) still carry full statistical power.
